## M1 - Problem Statement

Parse loan applications and financial statements into validated Pydantic LoanApplication / FinancialSnapshot objects via a provider-agnostic client.

In [ ]:
import os, json, time, sys
from typing import Literal, List
from pydantic import BaseModel, ValidationError

from dotenv import load_dotenv
load_dotenv(".env", override=True)

from langchain_litellm import ChatLiteLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-lite-latest")
chat_model = ChatLiteLLM(model=LLM_MODEL, temperature=0, api_key=os.getenv("GEMINI_API_KEY", GEMINI_API_KEY))

c:\Users\bharath-asus\Documents\LEARNING\Documentation\Articles\agentic_ai_projects\wealth_pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## read the jsonl file - the loan application records

with open("../data/wealthpilot/intake/records.jsonl") as f:
    records = [json.loads(line) for line in f]
print(f"Loaded {len(records)} records from records.jsonl.")

Loaded 200 records from records.jsonl.


In [3]:
# Define Pydantic models

class DeclaredFinancials(BaseModel):
    annual_revenue_inr: int | None = None
    ebitda_inr: int | None = None
    existing_debt_inr: int
    existing_annual_debt_service_inr: int | None = None
    current_assets_inr: int | None = None
    current_liabilities_inr: int | None = None
    months_operating: int | None = None
    collateral_value_inr: int

class GroundTruth(BaseModel):
    dscr: float | None = None
    meets_dscr_floor: bool | None = None
    risk_band: Literal["approve", "borderline", "decline"] | None = None
    requires_human_signoff: bool | None = None
    missing_fields: List[str] | None = None

class LoanApplication(BaseModel):
    application_id: str
    applicant_id: str
    channel: str
    received_at: str
    raw_narrative: str
    business_name: str | None = None
    sector: str | None = None
    requested_amount_inr: float
    requested_tenor_months: int
    declared_financials: DeclaredFinancials
    bureau_score: int | None = None
    ground_truth: GroundTruth
    record_id: str | None = None

In [5]:
RAW_INTAKE = """{"application_id": "ASH-L-00076", "applicant_id": "ASH-A-00076", "channel": "web_form",
"received_at": "2026-07-24T11:56",
"raw_narrative": "At Douglas Industries, we have been supplying precision auto parts to tier-1 makers 
for nearly two decades now out here in Malegaon. My family has always been deeply rooted in the Jain community,
attending the main temple on MG Road every single Tuesday, and we pride ourselves on absolute honesty in business.
We are looking for 1,700,000 over 48 months to fund our new CNC machinery purchase which will boost our output.
Our annual revenue stands at 85,000,000 rupees.",
"business_name": "Douglas Industries",
"sector": "Auto Components",
"requested_amount_inr": 1700000, "requested_tenor_months": 48, 
"declared_financials": {"annual_revenue_inr": 85000000, "ebitda_inr": 10884453, "existing_debt_inr": 17983173,
"existing_annual_debt_service_inr": 2423000, "current_assets_inr": 16358000,
"current_liabilities_inr": 9068000, "months_operating": 220, "collateral_value_inr": 2316000}, 
"bureau_score": 594,
"ground_truth": {"dscr": 3.65, "meets_dscr_floor": true, "risk_band": "approve", "requires_human_signoff": true, "missing_fields": []},
"record_id": "WEALTHPILOT-00000"}"""

In [6]:
def extract(text: str) -> dict:
        try:
            parsed = chat_model.with_structured_output(LoanApplication).invoke(
                [SystemMessage(content="Extract the loan application fields exactly as they appear. If they are missing, set them to an empty string."),
                 HumanMessage(content=text)])
            return parsed.model_dump()
        except Exception as e:
            print(f"  [structured extraction failed: {type(e).__name__}] using fixture")

In [7]:
extract(RAW_INTAKE)

{'application_id': 'ASH-L-00076',
 'applicant_id': 'ASH-A-00076',
 'channel': 'web_form',
 'received_at': '2026-07-24T11:56',
 'raw_narrative': 'At Douglas Industries, we have been supplying precision auto parts to tier-1 makers for nearly two decades now out here in Malegaon. My family has always been deeply rooted in the Jain community, attending the main temple on MG Road every single Tuesday, and we pride ourselves on absolute honesty in business. We are looking for 1,700,000 over 48 months to fund our new CNC machinery purchase which will boost our output. Our annual revenue stands at 85,000,000 rupees.',
 'business_name': 'Douglas Industries',
 'sector': 'Auto Components',
 'requested_amount_inr': 1700000.0,
 'requested_tenor_months': 48,
 'declared_financials': {'annual_revenue_inr': 85000000,
  'ebitda_inr': 10884453,
  'existing_debt_inr': 17983173,
  'existing_annual_debt_service_inr': 2423000,
  'current_assets_inr': 16358000,
  'current_liabilities_inr': 9068000,
  'months_

## Validation

In [ ]:
# Validate the record
def parse_loan_appl(raw: str) -> tuple[LoanApplication | None, str | None]:
    try:
        if isinstance(raw, str):
            raw = json.loads(raw)
        loan_application = LoanApplication(**raw)
        return loan_application, None
    except (json.JSONDecodeError, ValidationError) as e:
        return None, str(e)

In [10]:
loan_application, error = parse_loan_appl(extract(RAW_INTAKE))
loan_application

LoanApplication(application_id='ASH-L-00076', applicant_id='ASH-A-00076', channel='web_form', received_at='2026-07-24T11:56', raw_narrative='At Douglas Industries, we have been supplying precision auto parts to tier-1 makers for nearly two decades now out here in Malegaon. My family has always been deeply rooted in the Jain community, attending the main temple on MG Road every single Tuesday, and we pride ourselves on absolute honesty in business. We are looking for 1,700,000 over 48 months to fund our new CNC machinery purchase which will boost our output. Our annual revenue stands at 85,000,000 rupees.', business_name='Douglas Industries', sector='Auto Components', requested_amount_inr=1700000.0, requested_tenor_months=48, declared_financials=DeclaredFinancials(annual_revenue_inr=85000000, ebitda_inr=10884453, existing_debt_inr=17983173, existing_annual_debt_service_inr=2423000, current_assets_inr=16358000, current_liabilities_inr=9068000, months_operating=220, collateral_value_inr=2

In [13]:
if loan_application is not None:
    print("Validation successful.")
else:
    print(f"Validation failed with error: {error}")

Validation successful.


As the given data itself in the JSON format. Skipping the LLM extraction step for now, we will focus on validating the data and converting it into Pydantic objects.

In [44]:
results = []

for i in range(len(records)):
    loan_application, error = parse_loan_appl(records[i])
    results.append({"id":i, "loan_application": loan_application, "errors": error})

success = sum(1 for r in results if r["loan_application"] is not None)
print(f"Parsed {success}/{len(results)} records successfully.")


Parsed 200/200 records successfully.


In [45]:
## Listing down the records that failed validation, if any
[r for r in results if r["errors"] is not None]

[]